In [1]:
!pip install pyoxigraph
!pip install tqdm

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
data_path = '../../data/raw/dblp.ttl'

In [3]:
from pyoxigraph import Store, RdfFormat
from tqdm import tqdm

# Persistent database stored on disk
store = Store("../../data/raw/dblp_store")

In [4]:


query = """
PREFIX dblp: <https://dblp.org/rdf/schema#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?venue ?type ?name
WHERE {
    ?type rdfs:subClassOf+ dblp:Stream .

    ?venue rdf:type ?type .

    OPTIONAL {
        ?venue dblp:primaryStreamTitle ?name
    }
}
"""

results = store.query(query)

conferences_data = {}
journal_data = {}

for row in tqdm(results, total=9372):
    data = {
        "name": row['name'].value
    }

    if row['type'].value == "https://dblp.org/rdf/schema#Conference":
        conference_id = row['venue'].value.replace('https://dblp.org/streams/conf/', '')
        assert conference_id not in conferences_data, f"Duplicate conference venue found: {row['venue'].value}"
        conferences_data[conference_id] = data
    
    elif row['type'].value == "https://dblp.org/rdf/schema#Journal":
        journal_id = row['venue'].value.replace('https://dblp.org/streams/journals/', '')
        assert journal_id not in journal_data, f"Duplicate journal venue found: {row['venue'].value}"
        journal_data[journal_id] = data

print("5 Conferences examples:")
for i, (venue, data) in enumerate(conferences_data.items()):
    if i >= 5:
        break
    print(f"{venue}: {data}")

print('\n')

print("5 Journals examples:")
for i, (venue, data) in enumerate(journal_data.items()):
    if i >= 5:
        break
    print(f"{venue}: {data}")

100%|█████████▉| 9338/9372 [00:00<00:00, 10675.38it/s]

5 Conferences examples:
isami: {'name': 'International Symposium on Ambient Intelligence (ISAmI)'}
eds: {'name': 'Expert Database Systems (EDS)'}
ithgc: {'name': 'International Tourism, Hospitality & Gastronomy Congress (ITHGC)'}
imla: {'name': 'Workshop on Intuitionistic Modal Logic and Applications (IMLA)'}
ifip2: {'name': 'Central and East European Conference on Software Engineering Techniques (CEE-SET)'}


5 Journals examples:
apjor: {'name': 'Asia-Pacific Journal of Operational Research'}
tvcg: {'name': 'IEEE Transactions on Visualization and Computer Graphics (TVCG)'}
ijait: {'name': 'International Journal on Artificial Intelligence Tools (IJAIT)'}
ijst: {'name': 'International Journal of Speech Technology'}
iee-sej: {'name': 'Software Engineering Journal'}
